# Prep: clean + feature-select the datasets

Runs cleaning and feature selection for every dataset in `DATASETS` in one go, then **Save Version**. The reproduce/novel notebooks add this notebook's output as an input and skip cleaning.

Setup: turn Internet ON and attach the raw data for every dataset listed in `DATASETS` (drop a dataset from the list if you haven't attached it).

In [ ]:
DATASETS = ["nsl_kdd", "unsw_nb15", "cic_ids2017", "cse_cic_ids2018"]

# File patterns (searched recursively under /kaggle/input) that make up each dataset's raw files
PATTERNS = {
    "nsl_kdd": ["KDDTrain+*", "KDDTest+*"],
    "unsw_nb15": ["UNSW_NB15_training-set.csv", "UNSW_NB15_testing-set.csv"],
    "cic_ids2017": ["*WorkingHours*.csv"],
    "cse_cic_ids2018": ["*-2018.csv"],
}

In [ ]:
import glob, os, shutil, subprocess, sys

WORK = "/kaggle/working"
os.environ["RIS_DATA_RAW"] = f"{WORK}/data/raw"
os.environ["RIS_DATA_PROCESSED"] = f"{WORK}/data/processed"
os.environ["RIS_RESULTS"] = f"{WORK}/results"

REPO = f"{WORK}/RIS-M26"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Sahoo-Achyutananda/RIS-M26.git", REPO], check=True)
SRC = f"{REPO}/src"

missing = []
for ds in DATASETS:
    dest = f"{WORK}/data/raw/{ds}"
    os.makedirs(dest, exist_ok=True)
    found = []
    for pat in PATTERNS[ds]:
        found += glob.glob(f"/kaggle/input/**/{pat}", recursive=True)
    if not found:
        missing.append(ds)
        continue
    for f in sorted(set(found)):
        shutil.copy(f, dest)
    print(f"{ds}: copied {len(set(found))} files")

if missing:
    print("No raw files found for:", missing, "-> attach them or remove them from DATASETS. Files visible under /kaggle/input:")
    for root, _, files in os.walk("/kaggle/input"):
        for fn in files:
            print("  ", os.path.join(root, fn))
    raise AssertionError(f"raw data missing for {missing}")

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
for ds in DATASETS:
    for script in ("preprocess.py", "feature_selection.py"):
        print("=== " + ds + ": " + script + " ===", flush=True)
        proc = subprocess.run([sys.executable, "-u", script, "--dataset", ds],
                              cwd=SRC, env=os.environ, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(proc.stdout[-2500:])
        assert proc.returncode == 0, f"{ds}: {script} failed"

In [ ]:
import pandas as pd
for ds in DATASETS:
    d = pd.read_csv(f"{WORK}/data/processed/{ds}/dataset_final.csv")
    print(f"{ds}: {d.shape}, attack share {d['label'].mean():.3f}")